In [32]:
import pandas as pd
import numpy as np
from pathlib import Path
import os
import sys
from glob import glob
import pypower
from pypower import idx_bus
from pypower import idx_gen
from pypower import idx_brch
from pypower import idx_cost
import pypsa

In [8]:
bus_idxs = [atr for atr in dir(idx_bus) if '__' not in atr]

In [9]:
bus_col_map = {atr:getattr(pypower.idx_bus, atr) for atr in bus_idxs}
bus_col_map

{'BASE_KV': 9,
 'BS': 5,
 'BUS_AREA': 6,
 'BUS_I': 0,
 'BUS_TYPE': 1,
 'GS': 4,
 'LAM_P': 13,
 'LAM_Q': 14,
 'MU_VMAX': 15,
 'MU_VMIN': 16,
 'NONE': 4,
 'PD': 2,
 'PQ': 1,
 'PV': 2,
 'QD': 3,
 'REF': 3,
 'VA': 8,
 'VM': 7,
 'VMAX': 11,
 'VMIN': 12,
 'ZONE': 10}

In [10]:
data_path = Path("../case_data/")
files = glob(str(data_path/"*"))
files

['..\\case_data\\Midwest24k_20220923.m',
 '..\\case_data\\Wisconsin_1664.m',
 '..\\case_data\\Wisconsin_1664.mat']

In [11]:
case_path = Path(files[1])
case_path

WindowsPath('../case_data/Wisconsin_1664.m')

In [33]:
with open(case_path, "r") as file:
    data = file.readlines()

In [46]:
pypsa_comps = {}

print(len(data))
metadata = {}
start_row = []
for n, line in enumerate(data):
    if ("%%" in line):
        print(n)
        obj_name = line.replace("data", "").replace('%%','').strip()
        print(obj_name)
        metadata[obj_name] = n
        start_row.append(n)

start_row.append(len(data))

6138
5
bus
1673
generator
1756
generator cost
1839
branch
4305
bus names
5973
Generator Unit Types
6056
Generator Fuel Types


In [54]:
components = {}
for n, (name, row) in enumerate(metadata.items()):
    print(n, name, row)
    print(f"Streaming {start_row[n+1]-row-4} rows")
    comp = pd.read_csv(case_path, sep=r'\s+', skiprows=row+2, nrows=start_row[n+1]-row-4, header=None)
    display(comp.head(6))
    display(comp.tail(6))

0 bus 5
Streaming 1664 rows


,0,1,2,3,4,5,6,7,8,9,10,11,12
0,1,3,4.35,1.74,0.0,0.0,1,1.020000,0.000000,69.0,1,1.1,0.98
1,2,2,3.04,1.28,0.0,0.0,1,1.020003,-12.263547,69.0,1,1.1,0.98
2,3,2,64.92,21.98,0.0,0.0,1,1.020003,-7.580765,69.0,1,1.1,0.98
3,4,2,10.90,3.92,0.0,0.0,1,1.020002,-15.675466,69.0,1,1.1,0.98
4,5,2,3.33,1.50,0.0,0.0,1,1.020002,-14.138741,69.0,1,1.1,0.98
5,6,2,6.97,3.10,0.0,0.0,1,1.020001,-18.150550,69.0,1,1.1,0.98


,0,1,2,3,4,5,6,7,8,9,10,11,12
1658,1659,1,0.0,0.0,0.0,0.0,1,1.010679,-16.620172,138.0,1,1.1,0.9
1659,1660,1,0.0,0.0,0.0,0.0,1,1.010741,-16.665122,230.0,1,1.1,0.9
1660,1661,1,0.0,0.0,0.0,0.0,1,1.006129,-17.128617,230.0,1,1.1,0.9
1661,1662,1,0.0,0.0,0.0,0.0,1,0.972473,-29.700031,138.0,1,1.1,0.9
1662,1663,1,0.0,0.0,0.0,0.0,1,0.971578,-29.949311,138.0,1,1.1,0.9
1663,1664,1,0.0,0.0,0.0,0.0,1,0.969482,-30.594938,138.0,1,1.1,0.9


1 generator 1673
Streaming 79 rows


,0,1,2,3,4,5,6,7,8,9,...,11,12,13,14,15,16,17,18,19,20
0,1,1214.97,-27.89,466.80,-422.70,1.02,100.0,1,1286.0,300.0,...,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0.0
1,2,892.20,179.91,683.20,-407.60,1.02,100.0,1,1240.0,480.0,...,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0.0
2,3,657.50,209.37,764.30,-405.30,1.02,100.0,1,1233.2,600.0,...,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0.0
3,4,280.00,52.57,374.55,-198.65,1.02,100.0,1,604.4,280.0,...,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0.0
4,4,280.00,52.57,374.55,-198.65,1.02,100.0,1,604.4,280.0,...,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0.0
5,5,895.80,-76.37,538.60,-365.50,1.02,100.0,1,1112.0,300.0,...,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0.0


...

,0
0,'NP';
1,'ST';
2,'ST';
3,'CC';
4,'CC';
5,'ST';


,0
73,'ST';
74,'HY';
75,'ST';
76,'HY';
77,'ST';
78,'WT';


6 Generator Fuel Types 6056
Streaming 78 rows


,0
0,'nuclear';
1,'coal';
2,'coal';
3,'ng';
4,'ng';
5,'coal';


,0
72,'hydro';
73,'ng';
74,'hydro';
75,'wood';
76,'hydro';
77,'wood';


## Extract bus data

In [ ]:
bus_data = pd.read_csv(case_path, skiprows=7, sep=r'\s+', nrows=1664, header=None)

In [14]:
bus_header_map = {
pypower.idx_bus.BASE_KV:'BASE_KV',
pypower.idx_bus.BS:'BS',
pypower.idx_bus.BUS_AREA:'BUS_AREA',
pypower.idx_bus.BUS_I:'BUS_I',
pypower.idx_bus.BUS_TYPE:'BUS_TYPE',
pypower.idx_bus.GS:'GS',
pypower.idx_bus.LAM_P:'LAM_P',
pypower.idx_bus.LAM_Q:'LAM_Q',
pypower.idx_bus.MU_VMAX:'MU_VMAX',
pypower.idx_bus.MU_VMIN:'MU_VMIN',
# pypower.idx_bus.NONE:'NONE',
pypower.idx_bus.PD:'PD',
# pypower.idx_bus.PQ:'PQ',
# pypower.idx_bus.PV:'PV',
pypower.idx_bus.QD:'QD',
# pypower.idx_bus.REF:'REF',
pypower.idx_bus.VA:'VA',
pypower.idx_bus.VM:'VM',
pypower.idx_bus.VMAX:'VMAX',
pypower.idx_bus.VMIN:'VMIN',
pypower.idx_bus.ZONE:'ZONE'
}

In [15]:
bus_data.rename(columns=bus_header_map)

,BUS_I,BUS_TYPE,PD,QD,GS,BS,BUS_AREA,VM,VA,BASE_KV,ZONE,VMAX,VMIN
0,1,3,4.35,1.74,0.0,0.0,1,1.020000,0.000000,69.0,1,1.1,0.98
1,2,2,3.04,1.28,0.0,0.0,1,1.020003,-12.263547,69.0,1,1.1,0.98
2,3,2,64.92,21.98,0.0,0.0,1,1.020003,-7.580765,69.0,1,1.1,0.98
3,4,2,10.90,3.92,0.0,0.0,1,1.020002,-15.675466,69.0,1,1.1,0.98
4,5,2,3.33,1.50,0.0,0.0,1,1.020002,-14.138741,69.0,1,1.1,0.98
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1659,1660,1,0.00,0.00,0.0,0.0,1,1.010741,-16.665122,230.0,1,1.1,0.90
1660,1661,1,0.00,0.00,0.0,0.0,1,1.006129,-17.128617,230.0,1,1.1,0.90
1661,1662,1,0.00,0.00,0.0,0.0,1,0.972473,-29.700031,138.0,1,1.1,0.90
1662,1663,1,0.00,0.00,0.0,0.0,1,0.971578,-29.949311,138.0,1,1.1,0.90


## Extract Generator Data

In [16]:
gen_idxs = [atr for atr in dir(idx_gen) if '__' not in atr]
gen_col_map = {atr:getattr(idx_gen, atr) for atr in gen_idxs}
gen_col_map

{'APF': 20,
 'GEN_BUS': 0,
 'GEN_STATUS': 7,
 'MBASE': 6,
 'MU_PMAX': 21,
 'MU_PMIN': 22,
 'MU_QMAX': 23,
 'MU_QMIN': 24,
 'PC1': 10,
 'PC2': 11,
 'PG': 1,
 'PMAX': 8,
 'PMIN': 9,
 'QC1MAX': 13,
 'QC1MIN': 12,
 'QC2MAX': 15,
 'QC2MIN': 14,
 'QG': 2,
 'QMAX': 3,
 'QMIN': 4,
 'RAMP_10': 17,
 'RAMP_30': 18,
 'RAMP_AGC': 16,
 'RAMP_Q': 19,
 'VG': 5}

In [22]:
gen_header_map = {v:k for k,v in gen_col_map.items()}
gen_header_map

{20: 'APF',
 0: 'GEN_BUS',
 7: 'GEN_STATUS',
 6: 'MBASE',
 21: 'MU_PMAX',
 22: 'MU_PMIN',
 23: 'MU_QMAX',
 24: 'MU_QMIN',
 10: 'PC1',
 11: 'PC2',
 1: 'PG',
 8: 'PMAX',
 9: 'PMIN',
 13: 'QC1MAX',
 12: 'QC1MIN',
 15: 'QC2MAX',
 14: 'QC2MIN',
 2: 'QG',
 3: 'QMAX',
 4: 'QMIN',
 17: 'RAMP_10',
 18: 'RAMP_30',
 16: 'RAMP_AGC',
 19: 'RAMP_Q',
 5: 'VG'}

In [30]:
gen_data = pd.read_csv(case_path, skiprows=1675, sep=r'\s+', nrows=1752-1673, header=None)
gen_data.rename(columns=gen_header_map, inplace=True)